In [ ]:
import numpy as np
import pandas as pd
import Create_Lattices as cl
import matplotlib.pyplot as plt

BoxLengths = np.arange(50, 100, 2)
rhos = np.arange(0.2, 1.0, 0.05)

rows = []

for lattice_type in ["cubic", "fcc"]:
    for L in BoxLengths:
        for rho in rhos:
            if lattice_type == "cubic":
                pos = cl.FillBoxCubicLattice(L, rho)
            elif lattice_type == "fcc":
                pos = cl.FillBoxFccLattice(L, rho)

            actual_rho = len(pos) / L**3

            rows.append({
                "lattice_type": lattice_type,
                "BoxLength": L,
                "rho_target": rho,
                "N": len(pos),
                "rho_actual": actual_rho,
                "rho_error": actual_rho - rho,
                "abs_rho_error": abs(actual_rho - rho),
                "relative_error": abs(actual_rho - rho) / rho,
            })

df = pd.DataFrame(rows)

summary = (
    df.groupby("lattice_type")
      [["abs_rho_error", "relative_error"]]
      .agg(["mean", "median", "max"])
)

display(summary)





all_errors = df["relative_error"]

bins = np.linspace(
    all_errors.min(),
    all_errors.max(),
    41,   # 40 bins
)

plt.figure(figsize=(8, 5))

for lattice_type in ["cubic", "fcc"]:
    sub = df[df["lattice_type"] == lattice_type]

    plt.hist(
        sub["relative_error"],
        bins=bins,
        alpha=0.6,
        label=lattice_type,
    )

plt.xlabel("Relative density error")
plt.ylabel("Count")
plt.legend()
plt.title("Density accuracy: cubic vs FCC")
plt.show()

In [ ]:
mean_error_vs_L = (
    df.groupby(["lattice_type", "BoxLength"])
      ["relative_error"]
      .mean()
      .unstack()
      .T
)

mean_error_vs_L.plot(marker="o")
plt.ylabel("Mean relative error")
plt.grid()
plt.show()